<a href="https://colab.research.google.com/github/Narasimha722/achilles_AI-ML_Resource/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Achilles quickstart: Colab or Kaggle

This notebook sets up [Achilles](https://github.com/Lourdhu02/achilles) in about five minutes. It clones the repo, checks the hardware, runs one lab's reference tests, measures the accelerator and pretrains a small GPT.

**GPU (optional):** go to **Runtime → Change runtime type → T4 GPU**. Every cell also works on a CPU runtime, just more slowly.

> The runtime's disk is wiped when the session ends. Save your `exercise.py` work to a fork or to Google Drive before you close the tab.

In [1]:
# 1. Hardware check
!nvidia-smi -L || echo "No NVIDIA GPU: running on CPU (fine for every lab's tests)"

GPU 0: Tesla T4 (UUID: GPU-52ac3017-357b-4c41-f462-adc64bfefbd3)


In [2]:
# 2. Clone and install (PyTorch is preinstalled on Colab and Kaggle, so torch is not reinstalled)
import os
if not os.path.exists("achilles"):
    !git clone --depth 1 https://github.com/Lourdhu02/achilles.git
%cd achilles
!pip install -q -r requirements.txt

Cloning into 'achilles'...
remote: Enumerating objects: 359, done.
remote: Counting objects: 100% (359/359), done.
remote: Compressing objects: 100% (330/330), done.
remote: Total 359 (delta 25), reused 251 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (359/359), 2.59 MiB | 2.49 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/achilles
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.6 MB/s eta 0:00:00


In [3]:
# 3. Which backend will the labs use?
import torch
print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))
else:
    print("CPU only")

torch 2.11.0+cu130
GPU: Tesla T4 | compute capability (7, 5)


In [4]:
# 4. Run one lab's reference tests (about a minute)
!python -m pytest labs/05_transformer --impl=solution -q

.................                                                        [100%]
17 passed in 10.87s


In [5]:
# 5. Measure the roofline: matmul TFLOP/s, memory bandwidth, and your hardware tier
!python tools/measure_gpu.py

device: Tesla T4  compute capability 7.5  memory 14.6 GiB  SMs 40

dtype                 n=1024      n=2048      n=4096      n=8192   (dense matmul TFLOP/s)
fp32 (no TF32)          3.97        4.43        4.51        4.31
tf32                    3.73        4.29        4.39        4.26
bf16                    2.90        2.97        2.63        2.60
fp16                   36.22       42.14       41.16       24.71

memory bandwidth (device copy):    235.5 GB/s
bfloat16 mat-vec n=8192: 0.257 TFLOP/s, streaming weights at 257.3 GB/s  <- decode is memory-bound

ridge point (bf16): 13 FLOP/byte -- kernels below this intensity are memory-bound
Now predict: batch-1 decode tokens/s for a 0.5B bf16 model and an 8B 4-bit model (lab 03).

hardware tier: NVIDIA 8-16 GB (15 GiB) -> train.py --preset gpu-8gb


## Pretrain a small GPT on TinyStories

This downloads the TinyStories validation split (about 20 MB) and trains the reference model from lab 05 for a few hundred steps. On a T4, `train.py` uses fp16 with loss scaling automatically, because the T4 has no fast bf16.

Swap `--impl solution` for `--impl exercise` once your own `labs/05_transformer/exercise.py` passes its tests.

In [ ]:
# 6. Data (the file goes to data/, which git ignores)
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
path = hf_hub_download("roneneldan/TinyStories", "TinyStoriesV2-GPT4-valid.txt", repo_type="dataset", local_dir="data")
print(path)

In [ ]:
# 7. Train: the preset picks model and batch sizes for this hardware
preset = "gpu-8gb" if torch.cuda.is_available() else "cpu"
!python labs/05_transformer/train.py --data data/TinyStoriesV2-GPT4-valid.txt --impl solution --preset {preset} --max-steps 500

## Next: your first lab

```bash
!python -m pytest labs/01_autograd          # 37 failing tests: your first job
!python tools/progress.py                   # your scoreboard across all 18 labs
```

Open `labs/01_autograd/README.md` for the handout and edit `labs/01_autograd/exercise.py` in the file browser on the left. The full learning path is in the [README](https://github.com/Lourdhu02/achilles#readme).